In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/ayomidefagbolade/combined-df/combined_data.csv


In [2]:
#!pip install vllm

In [3]:
#!pip install "protobuf<6.0.0" --force-reinstall

In [4]:
import pandas as pd

# Read the CSV file into df
df = pd.read_csv('/kaggle/input/datasets/ayomidefagbolade/combined-df/combined_data.csv')

# Display the first few rows
df .head()

,combined_text
0,sorghum low local production of grains
1,banana/plantain can be used as a remedy for co...
2,"sorghum prices decreased in many regions, with..."
3,banana/plantain Banana Bacteria Wilt disease i...
4,cassava funding for production


In [5]:
import json
import re
import pandas as pd
from vllm import LLM, SamplingParams


In [6]:

TEXT_COLUMN = "combined_text"                # column containing the text to classify
OUTPUT_PATH = "combined_df_labeled.csv"
BATCH_SIZE = 100                     # prompts sent to llm.generate() at once
MAX_NEW_TOKENS = 300

In [7]:
TAXONOMY = {
    "1a": "Market/Trade Challenges (poor access, low prices, import dependency, trade barriers)",
    "1b": "Market Development & Linkages (export promotion, market-access programs, value-chain integration)",
    "2a": "Processing Development (technology, facilities, commercialization)",
    "2b": "Bioenergy & Byproduct Use (ethanol, biofuel, waste conversion)",
    "3a": "Policy & Regulation (laws, trade policy, biosafety, insurance frameworks)",
    "3b": "Government Funding & Grants (subsidies, grant programs, funding allocation)",
    "4a": "Production Challenges (poor yields, production deficits)",
    "4b": "Production Improvement & Mechanization (better practices, machinery adoption, production targets)",
    "5a": "Input/Seed Shortages & Quality Issues (scarcity, wrong variety delivered, quality failure)",
    "5b": "Input Supply & Seed Systems (distribution, certification, multiplication)",
    "6a": "Genetic Vulnerability (disease susceptibility, genetic bottleneck, limited diversity)",
    "6b": "Breeding & Variety Development (resistance breeding, biotech, diversity conservation)",
    "7a": "Climate Risk & Crop Vulnerability (drought damage, erratic rainfall, crop failure)",
    "7b": "Climate-Adaptive Practices (resilient variety selection, adaptive farming methods)",
    "8a": "Outbreaks & Infestation (disease/pest/weed damage, crop loss events)",
    "8b": "Control & Prevention (IPM, control programs, soil/rotation management)",
    "9a": "Post-Harvest Losses (spoilage, poor handling, transport damage)",
    "9b": "Storage & Handling Improvement (facility investment, technology, infrastructure)",
    "10a": "Credit & Financial Barriers (loan inaccessibility, lack of collateral)",
    "10b": "Financial Support Programs (grants, loans, cooperatives, insurance)",
    "11a": "Nutritional Deficiency (micronutrient/vitamin A deficiency, malnutrition)",
    "11b": "Biofortification & Nutrition Programs (breeding, fortification initiatives)",
    "12a": "Extension Service Delivery (training programs, advisory services, materials)",
    "13a": "R&D Programs & Funding (research initiatives, capacity, funding constraints)",
    "14a": "Income & Livelihood Outcomes (income increase, job creation)",
    "14b": "Economic Contribution (GDP impact, production rankings, sector growth)",
    "15a": "Diversification Initiatives (alternative crops, underutilized crops, risk-spreading programs)",
    "16a": "Data & Monitoring Systems (yield tracking, reporting, M&E infrastructure)",
    "17a": "Water Scarcity & Irrigation Deficits (drought stress, lack of irrigation)",
    "17b": "Irrigation Development (system installation, rehabilitation, water management)",
    "18a": "Quality Issues & Complaints (taste/quality complaints, non-compliance)",
    "18b": "Standards & Certification (standardization efforts, certification programs)",
    "19a": "Gender Disparity (inequality, exclusion of women farmers)",
    "19b": "Inclusion Programs (support for women farmers, equity initiatives)",
    "20a": "Food Insecurity & Land Access Barriers (staple crop shortages, land access issues, political vulnerability)",
    "20b": "Food Security & Land Programs (staple crop initiatives, land consolidation)",
}


TOPIC_NAMES = {
    1: "Market Access, Trade & Value Chains",
    2: "Agro-Processing & Value Addition",
    3: "Agricultural Policy, Regulation & Government Support",
    4: "Crop Production, Farming Practices & Mechanization",
    5: "Agricultural Inputs & Seed Systems",
    6: "Crop Breeding, Genetics & Stress Resistance",
    7: "Climate Adaptation",
    8: "Pest, Disease, Weed & Soil Management",
    9: "Storage, Post-Harvest Handling & Loss Reduction",
    10: "Financial Services & Smallholder Support",
    11: "Nutrition & Biofortification",
    12: "Extension Services & Farmer Training",
    13: "Research & Development",
    14: "Economic Impact & Employment",
    15: "Crop Diversification",
    16: "Agricultural Data & Yield Monitoring",
    17: "Irrigation & Water Management",
    18: "Crop Quality & Standards Compliance",
    19: "Gender & Social Inclusion",
    20: "Food Security & Land Access",
}

FEW_SHOT_EXAMPLES = """Examples (including short/terse passages — these still deserve labels
if the core theme is clear; do not require a full sentence or lots of detail to commit to a code):

Passage: "cassava production increased by 10% in 2015 compared to 2014"
Output: {"labels": ["4b"]}

Passage: "sorghum guaranteeing food security"
Output: {"labels": ["20b"]}

Passage: "exemplary farmers are achieving high yields of 50 quintals per hectare"
Output: {"labels": ["4b"]}

Passage: "banana/plantain is a potassium-rich fruit that can help reduce blood pressure, decrease bone loss, and prevent kidney stones"
Output: {"labels": ["11b"]}

Passage: "advice on proper farming methods and practices for cassava"
Output: {"labels": ["4b"]}

Passage: "the annual general meeting will be held next Tuesday at the district office"
Output: {"labels": []}
"""

ALL_CODES = list(TAXONOMY.keys())

def build_taxonomy_block() -> str:
    """Render taxonomy grouped by topic number for the prompt."""
    lines = []
    for topic_num, topic_name in TOPIC_NAMES.items():
        lines.append(f"{topic_num}. {topic_name}")
        for code, desc in TAXONOMY.items():
            if re.match(rf"^{topic_num}[ab]$", code):
                lines.append(f"   {code}: {desc}")
    return "\n".join(lines)


TAXONOMY_BLOCK = build_taxonomy_block()

In [8]:

SYSTEM_PROMPT = f"""You are an expert annotator classifying short agriculture-related text passages \
into a fixed multi-label taxonomy of subtopics. Each passage can belong to ONE OR MORE subtopics \
(multi-label), or none if nothing applies.

TAXONOMY (code: description):
{TAXONOMY_BLOCK}

Instructions:
- Read the passage carefully, including short or terse passages — brevity is not a reason to skip
  a label. If the passage clearly names a theme in the taxonomy (e.g. "production increased" →
  production improvement), label it even if there's no extra detail.
- Select every subtopic code that clearly applies. Most passages will have 1-3 labels.
- Only return an empty list when the passage truly has no connection to any taxonomy topic
  (e.g. purely administrative/logistical content, unrelated to agriculture).
- Respond with ONLY a JSON object, no other text, no markdown fences, in this exact form:
  {{"labels": ["<code>", "<code>", ...]}}
- Use only codes from the taxonomy above (e.g. "1a", "7b", "20a"). Do not invent new codes.

{FEW_SHOT_EXAMPLES}"""

# Used on the optional second pass for rows that came back with zero labels on pass 1.
# Pushes the model to commit to the single best-fitting code rather than defaulting to empty.
SECOND_PASS_SYSTEM_PROMPT = f"""You are an expert annotator. A first classification pass on this \
passage returned no labels. Look again and pick the SINGLE best-fitting subtopic code, even if the \
match is imperfect — only return an empty list if the passage is genuinely unrelated to agriculture \
or none of the codes are even loosely applicable.

TAXONOMY (code: description):
{TAXONOMY_BLOCK}

Respond with ONLY a JSON object: {{"labels": ["<code>"]}} or {{"labels": []}} if truly nothing fits.
"""


In [9]:
USER_TEMPLATE = "Passage:\n\"\"\"\n{text}\n\"\"\"\n\nReturn the JSON labels now."





In [10]:
def build_prompt(tokenizer, text: str, system_prompt: str = None) -> str:
    messages = [
        {"role": "system", "content": system_prompt or SYSTEM_PROMPT},
        {"role": "user", "content": USER_TEMPLATE.format(text=text)},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )



In [11]:
def parse_labels(raw_output: str) -> list:
    """Extract a valid list of taxonomy codes from the model's raw output."""
    raw_output = raw_output.strip()
    # Strip accidental markdown fences
    raw_output = re.sub(r"^```(json)?|```$", "", raw_output, flags=re.MULTILINE).strip()

    labels = []
    try:
        obj = json.loads(raw_output)
        labels = obj.get("labels", [])
    except json.JSONDecodeError:
        # Fallback: pull out anything that looks like a valid code, e.g. "7b"
        found = re.findall(r'"?(\d{1,2}[ab])"?', raw_output)
        labels = found

    # Keep only valid, deduplicated codes
    clean = []
    for l in labels:
        l = str(l).strip().lower()
        if l in TAXONOMY and l not in clean:
            clean.append(l)
    return clean


In [12]:
def classify_texts(llm, tokenizer, texts: list, sampling_params, system_prompt: str) -> list:
    """Run a batched classification pass over a list of texts and return a
    list of label-lists (one per input text, same order)."""
    prompts = [build_prompt(tokenizer, t, system_prompt) for t in texts]

    results = [None] * len(prompts)
    for start in range(0, len(prompts), BATCH_SIZE):
        end = min(start + BATCH_SIZE, len(prompts))
        batch = prompts[start:end]
        outputs = llm.generate(batch, sampling_params)
        for i, out in enumerate(outputs):
            raw = out.outputs[0].text
            results[start + i] = parse_labels(raw)
        print(f"  ...processed {end}/{len(prompts)}")
    return results

In [13]:
texts = df[TEXT_COLUMN].fillna("").astype(str).tolist()
print(f"Loaded {len(texts)} rows from comnined_df")


Loaded 18952 rows from comnined_df


In [14]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("huggingface")
os.environ["HF_TOKEN"] = secret_value_0

In [15]:
llm = LLM(
        model="meta-llama/Llama-3.1-8B-Instruct",
        dtype="half",
        tensor_parallel_size=2,
        gpu_memory_utilization=0.85,
        disable_custom_all_reduce=True,
        max_model_len=10000,
    )
tokenizer = llm.get_tokenizer()

sampling_params = SamplingParams(
        temperature=0.01,      # deterministic classification
        max_tokens=MAX_NEW_TOKENS,
    )

INFO 07-22 15:23:28 [api_utils.py:273] non-default args: {'dtype': 'half', 'max_model_len': 10000, 'tensor_parallel_size': 2, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'disable_custom_all_reduce': True, 'model': 'meta-llama/Llama-3.1-8B-Instruct'}


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

INFO 07-22 15:23:45 [model.py:619] Resolved architecture: LlamaForCausalLM
WARNING 07-22 15:23:45 [model.py:2114] Casting torch.bfloat16 to torch.float16.
INFO 07-22 15:23:45 [model.py:1776] Using max model len 10000
INFO 07-22 15:23:45 [scheduler.py:252] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 07-22 15:23:45 [vllm.py:1042] Asynchronous scheduling is enabled.
INFO 07-22 15:23:45 [kernel.py:292] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Capturing CUDA graphs (FULL):  97%|█████████▋| 34/35 [00:09<00:00,  3.55it/s]

(Worker_TP1 pid=314) 

Capturing CUDA graphs (FULL): 100%|██████████| 35/35 [00:11<00:00,  3.09it/s]

INFO 07-22 15:26:56 [model_runner.py:722] Graph capturing finished in 20 secs, took 1.22 GiB


(Worker_TP0 pid=313) INFO 07-22 15:26:56 [model_runner.py:722] Graph capturing finished in 20 secs, took 1.22 GiB
(Worker_TP0 pid=313) INFO 07-22 15:27:17 [jit_monitor.py:73] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.
(Worker_TP1 pid=314) INFO 07-22 15:27:17 [jit_monitor.py:73] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.
(EngineCore pid=295) INFO 07-22 15:27:18 [core.py:337] init engine (profile, create kv cache, warmup model) took 81.14 s (compilation: 31.86 s)
(EngineCore pid=295) INFO 07-22 15:27:19 [kernel.py:292] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
(Worker_TP0 pid=313) WARNING 07-22 15:27:22 [jit_monitor.py:129] Triton kernel JIT compilation during inference: kernel_unified_attention. This causes a latency spike; consider extending warmup to cover this shape/config.


In [16]:



all_labels = classify_texts(llm, tokenizer, texts, sampling_params, SYSTEM_PROMPT)


Rendering prompts:   0%|          | 0/100 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 100/100 [00:06<00:00, 14.76it/s, est. speed input: 20710.37 toks/s, output: 135.56 toks/s]

  ...processed 100/18952


Rendering prompts:   0%|          | 0/100 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 100/100 [00:04<00:00, 22.40it/s, est. speed input: 31422.27 toks/s, output: 201.65 toks/s]

...

Processed prompts: 100%|██████████| 100/100 [00:04<00:00, 21.58it/s, est. speed input: 30246.80 toks/s, output: 190.16 toks/s]

  ...processed 18900/18952


Rendering prompts:   0%|          | 0/52 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 52/52 [00:03<00:00, 17.11it/s, est. speed input: 24016.05 toks/s, output: 163.94 toks/s]

  ...processed 18952/18952


In [18]:
 #Pass 2: re-classify rows that came back empty, pushing for a
# best-effort single label instead of leaving them blank.
# ------------------------------------------------------------------
empty_idx = [i for i, labels in enumerate(all_labels) if not labels]
print(f"Pass 1 complete. {len(empty_idx)} rows returned zero labels — running pass 2 on those.")

if empty_idx:
    empty_texts = [texts[i] for i in empty_idx]
    second_pass_labels = classify_texts(
        llm, tokenizer, empty_texts, sampling_params, SECOND_PASS_SYSTEM_PROMPT
    )
    for idx, labels in zip(empty_idx, second_pass_labels):
        all_labels[idx] = labels
    still_empty = sum(1 for l in second_pass_labels if not l)
    print(f"Pass 2 complete. {still_empty} rows still have zero labels (likely genuinely off-topic).")



Pass 1 complete. 478 rows returned zero labels — running pass 2 on those.


Rendering prompts:   0%|          | 0/100 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 100/100 [00:04<00:00, 21.95it/s, est. speed input: 23000.30 toks/s, output: 214.49 toks/s]

  ...processed 100/478


Rendering prompts:   0%|          | 0/100 [00:00<?, ?it/s]

...

Processed prompts: 100%|██████████| 100/100 [00:03<00:00, 26.06it/s, est. speed input: 27342.11 toks/s, output: 249.17 toks/s]

  ...processed 400/478


Rendering prompts:   0%|          | 0/78 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 78/78 [00:03<00:00, 23.72it/s, est. speed input: 24904.80 toks/s, output: 254.93 toks/s]

  ...processed 478/478
Pass 2 complete. 359 rows still have zero labels (likely genuinely off-topic).


In [19]:
# ------------------------------------------------------------------
# Attach results to dataframe
# ------------------------------------------------------------------
df["predicted_labels"] = all_labels  # list of codes, e.g. ["4a", "7a"]
df["predicted_labels_str"] = df["predicted_labels"].apply(lambda ls: ";".join(ls))

# One binary column per subtopic code
for code in ALL_CODES:
    df[f"label_{code}"] = df["predicted_labels"].apply(lambda ls: int(code in ls))


In [20]:



# ------------------------------------------------------------------
# Save
# ------------------------------------------------------------------
if OUTPUT_PATH.endswith(".parquet"):
    df.to_parquet(OUTPUT_PATH, index=False)
else:
    df.to_csv(OUTPUT_PATH, index=False)

print(f"Saved labeled dataframe to {OUTPUT_PATH}")





Saved labeled dataframe to combined_df_labeled.csv
